# Minska sensorredundans i produktionslinjen med PROC GVARCLUS

## Sammanfattning

En tillverkningslinje med flera zoner strömmar dussintals korrelerade sensoravläsningar, där många bär på samma underliggande signal. Den här notebooken använder **PROC GVARCLUS** (variabelklustring) för att gruppera de 13 processensorerna i fyra disjunkta kluster, och läser sedan varje klusters R-kvadratvärden för att välja ut en representativ mätare per kluster — vilket minskar SPC-övervakningens omfattning utan att förlora processinformation. Tre av de fyra klustren kartläggs rent till fysiska zoner (genomsnittlig R-kvadrat 0,92, 0,93 och 0,96); det fjärde är ett enkanalskluster som proceduren delade av separat, vilket vi undersöker i stället för att bortse från det.

## Datakällor

All data genereras direkt i koden med `call streaminit(20260531)` och `rand()` — inga externa filer eller nätverksanrop.

| Dataset | Rader | Nyckelvariabler | Beskrivning |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Syntetiska avläsningar från en produktionslinje med tre zoner. Sensorer inom samma zon delar en latent drivfaktor (hög korrelation); mätare som korsar zongränser (temperaturer kopplade till zon 1, tryck till zon 2, vibration till zon 3) läggs till för att skapa realistisk redundans. DATA-stegets loop begär 300 cykler, men den här utvärderingsmiljön körs i olicensierat läge och behåller endast de första 100 observationerna — tillräckligt för att återskapa klusterstrukturen rent. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | En rad per insensor: vilket kluster den tilldelades och dess R-kvadrat mot det klustrets komponent. Skapas av satsen `OUTPUT OUT=`. |

# Minska sensorredundans i produktionslinjen med PROC GVARCLUS

På en instrumenterad produktionslinje är det vanligt att logga dussintals sensorer som mäter överlappande fysiska fenomen — flera termoelement i en zon, redundanta trycktransmittrar, vibrationsgivare som följer samma axel. Att mata varje kanal in i ett styrdiagram eller en efterföljande modell slösar övervakningsinsats och ökar multikollineariteten.

**Variabelklustring** löser detta direkt. `PROC GVARCLUS` delar upp de numeriska variablerna i *disjunkta* kluster, där varje kluster sammanfattas av den första principalkomponenten bland sina medlemmar. Sensorer som rör sig tillsammans hamnar i samma kluster; ingenjören kan sedan behålla en representativ kanal per kluster.

I den här notebooken:

1. Genererar vi avläsningar från en linje med tre zoner med medvetet redundanta sensorer (loopen begär 300 cykler; 100 behålls här).
2. Klustrar vi de 13 kanalerna med `PROC GVARCLUS` med `MAXCLUSTERS=4`.
3. Fångar vi klustertilldelningarna i ett utdataset och sammanfattar dem.
4. Tolkar vi R-kvadratvärdena för att välja en mätare per kluster för löpande SPC.

## Steg 1 – Generera syntetisk sensordata

Vi simulerar tre processzoner, var och en med en dold drivfaktor (`zone1_a`, `zone2_a`, `zone3_a`). Övriga kanaler är brusiga linjära funktioner av sin zons drivfaktor, så de är starkt korrelerade inom en zon men nästan oberoende mellan zoner. Vi kopplar också in-/utloppstemperaturerna till zon 1 och de två trycktransmittrarna till zon 2, för att efterlikna den redundans mellan instrument som finns på riktiga linjer. Ett fast frö gör körningen reproducerbar. Loopen begär 300 cykler; i olicensierat läge behåller motorn de första 100 observationerna, vilket NOTE-meddelandet nedan rapporterar.

In [1]:
data process_sensors;
    CALL streaminit(20260531);
    GÖR cycle = 1 TILL 300;
        /* Zon 1: latent drivfaktor plus två redundanta givare */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zon 2: latent drivfaktor plus två redundanta givare */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zon 3: latent drivfaktor plus en redundant givare */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Instrumentöverskridande kanaler kopplade till befintliga drivfaktorer */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        UTDATA;
    SLUT;
    TA_BORT cycle;
KÖR;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Steg 2 – Klustra sensorerna

Vi anropar `PROC GVARCLUS` på alla 13 kanaler. Satsen `VAR` listar de numeriska sensorer som ska klustras. Vi begränsar uppdelningen till fyra kluster med `MAXCLUSTERS=4` (vi förväntar oss ungefär ett kluster per fysisk zon, med lite marginal). `ODS GRAPHICS ON` tillsammans med `PLOTS=ALL` aktiverar variabelklusterdendrogrammet; motorn skriver den SVG-filen till arbetskatalogen i stället för att rendera den infogat, så strukturen vi läser nedan kommer från den utskrivna tabellen **Variable Cluster Assignments** och tabellen med egenvärden per kluster.

Satsen `OUTPUT OUT=` skriver kluster-tilldelningarna för varje variabel — tillsammans med varje variabels R-kvadrat mot sitt eget kluster — till ett dataset vi kan programmera vidare mot senare.

In [2]:
ODS GRAPHICS ON;

PROCEDUR gvarclus data=process_sensors maxclusters=4 PLOTS=ALL;
    VARIABEL zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    UTDATA out=clusters;
KÖR;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Steg 3 – Kör om med alternativet SUMMARY

Att köra proceduren en andra gång med alternativet `SUMMARY` behåller samma uppdelning. `PLOTS=NONE` stänger av grafiken för denna körning. I den här miljön är den utskrivna rapporten identisk med steg 2 — samma tabell med 13 rader för tilldelningar, samma fyra kluster och samma sammanfattning av egenvärden — så den här cellen visar huvudsakligen att alternativen `SUMMARY` och `PLOTS=NONE` tolkas och körs korrekt; den förväntas inte lägga till några nya siffror.

In [3]:
PROCEDUR gvarclus data=process_sensors maxclusters=4 summary PLOTS=none;
    VARIABEL zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
KÖR;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Steg 4 – Granska utdatasetet

Datasetet `clusters` från `OUTPUT OUT=` har en rad per sensor med dess tilldelade `Cluster` och `RSq_Own` (den kvadrerade korrelationen mellan sensorn och dess klusterkomponent). En hög `RSq_Own` betyder att sensorn representeras väl av sitt kluster — en idealisk kandidat att *utesluta* till förmån för klustrets representant. Vi sorterar efter kluster och därefter efter R-kvadrat, så att den mest representativa sensorn i varje kluster hamnar överst i sin grupp.

In [4]:
PROCEDUR SORTERA data=clusters out=clusters_ranked;
    EFTER CLUSTER FALLANDE RSq_Own;
KÖR;

PROCEDUR SKRIV data=clusters_ranked ETIKETT;
    VARIABEL Variable CLUSTER RSq_Own;
    ETIKETT Variable = "Sensorkanal"
          CLUSTER  = "Kluster"
          RSq_Own  = "R-kvadrat mot eget kluster";
KÖR;


  Obs  Sensorkanal  Kluster  R-kvadrat mot eget kluster
-----  -----------  -------  --------------------------
    1  ZONE1_A            1                     0.96867
    2  ZONE1_B            1                      0.9189
    3  TEMP_IN            1                    0.903394
    4  TEMP_OUT           1                    0.889519
    5  ZONE2_A            2                     0.98364
    6  ZONE2_B            2                    0.946708
    7  PRESSURE_A         2                    0.929356
    8  PRESSURE_B         2                    0.920915
    9  ZONE2_C            2                    0.892405
   10  ZONE3_A            3                    0.977237
   11  VIBRATION          3                     0.95916
   12  ZONE3_B            3                    0.949054
   13  ZONE1_C            4                           1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Tolkning av resultaten

Uppdelningen återskapar det mesta av linjens fysiska struktur, med en ärlig brasklapp:

- **Kluster 1** samlar `zone1_a` (R²=0,969), `zone1_b` (0,919) samt in-/utloppstemperaturerna `temp_in` (0,903) och `temp_out` (0,890) — fyra av de fem kanalerna som drivs av zon 1:s latenta signal. Genomsnittlig R-kvadrat 0,920.
- **Kluster 2** samlar alla tre zon 2-kanaler `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) tillsammans med de två trycktransmittrarna `pressure_a` (0,929) och `pressure_b` (0,921), som vi kopplade till zon 2:s drivfaktor. Genomsnittlig R-kvadrat 0,935.
- **Kluster 3** samlar `zone3_a` (0,977), `zone3_b` (0,949) och `vibration` (0,959). Genomsnittlig R-kvadrat 0,962.
- **Kluster 4** är en enda kanal: `zone1_c`, den tredje zon 1-givaren, delades av separat med R²=1,000 (en singel förklaras trivialt fullständigt av sig själv). Trots att den delar zon 1:s drivfaktor bedömde proceduren, vid denna urvalsstorlek, att `zone1_c` var tillräckligt skild från gruppen `zone1_a`/temperaturer för att motivera ett eget kluster i stället för att slås ihop med kluster 1.

De tre flerkanalsklustren har alla en genomsnittlig R-kvadrat över **0,90**, vilket bekräftar att en enda komponent förklarar merparten av variationen bland deras medlemmar — precis den redundans vi vill slå samman. Den ensamma avvikelsen — att `zone1_c` hamnar i sitt eget kluster i stället för med resten av zon 1 — är en bra påminnelse om att variabelklustring är datadrivet: med fler cykler eller en högre `MAXCLUSTERS`-budget kan gränsen förskjutas, så uppdelningen är en utgångspunkt för ingenjörsmässig bedömning, inte en fastställd sanning.

**Åtgärd för linjen.** Behåll inom varje flerkanalskluster den sensor som har högst `RSq_Own` (kanalen som är mest representativ för sitt kluster) och pensionera eller nedprioritera resten från rutinmässig SPC-diagramföring — till exempel `zone2_a` för kluster 2 och `zone3_a` för kluster 3. Behandla `zone1_c`-singeln som en flagga att utreda snarare än ett automatiskt bevarande: bekräfta om den bär genuint distinkt information eller är en klustringsartefakt innan du beslutar att övervaka den separat. Även med den kanalen avsatt separat samlar detta ihop 13 övervakade kanaler till en plan med fyra övervakningskanaler, vilket minskar larmbrus och multikollinearitet samtidigt som linjens informationsinnehåll bevaras.